# 實作 2: 使用 Tool 保存及讀取 ERP 工作備忘錄

In [1]:
# 建立自訂的 Static Context Scheme
from dataclasses import dataclass

@dataclass
class UserStaticContext:
    # 由 ERP 登入系統或應用程式提供的可信任識別碼
    user_id: str

In [2]:
# System Prompt
erp_memo_system_prompt = """
You are an ERP assistant.

Follow these rules:
- When the user asks you to remember or track an ERP document, call
  `save_erp_follow_up`.
- Preserve the ERP document ID provided by the user.
- When the user asks for saved ERP follow-up items, call
  `get_erp_follow_ups`.
- Base your answer on the tool result. Do not invent saved follow-up items.
"""

## Tools

### 建立寫入工作備忘錄的 Tool

In [3]:
from datetime import datetime

from langchain.tools import ToolRuntime, tool

@tool
def save_erp_follow_up(
    document_type: str,
    document_id: str,
    note: str,
    runtime: ToolRuntime[UserStaticContext]
) -> str:
    """
    Save a follow-up note for an ERP document.

    Args:
        document_type: The ERP document type, such as purchase_order.
        document_id: The ERP document ID provided by the user.
        note: The follow-up note that the user wants to remember.
        runtime: The ToolRuntime object injected by LangChain.

    Returns:
        A message indicating that the follow-up note was saved.
    """

    assert runtime.store is not None

    # 從可信任的 Agent Context 取得目前登入者
    user_id = runtime.context.user_id

    # 每一位 User 使用不同的 namespace
    namespace = ("users", user_id, "follow_ups")
    
    # 使用文件類型與文件編號識別一筆工作備忘錄
    key = f"{document_type}-{document_id}"

    # Save to the runtime store
    runtime.store.put(
        namespace,
        key,
        {
            "document_type": document_type,
            "document_id": document_id,
            "note": note
        }
    )
    current_timestamp = datetime.now().isoformat()
    return f"已保存 {document_id} 的追蹤事項。 ({current_timestamp})"

### 建立讀取工作備忘錄的 Tool 

In [4]:
@tool
def get_erp_follow_ups(
    runtime: ToolRuntime[UserStaticContext]
) -> list[dict]:
    """
    Get the current user's ERP follow-up notes.

    Args:
        runtime: The ToolRuntime object injected by LangChain.

    Returns:
        A list containing the current user's ERP follow-up notes.
    """

    assert runtime.store is not None

    user_id = runtime.context.user_id
    namespace = ("users", user_id, "follow_ups")

    items = runtime.store.search(namespace)

    return [
        {
            "memory_id": item.key,
            **item.value
        }
        for item in items
    ]

## 建立使用 Memory Store 的 ERP Agent

In [5]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

checkpointer = InMemorySaver()
store = InMemoryStore()

agent = create_agent(
    model="gpt-5-nano",
    tools=[save_erp_follow_up, get_erp_follow_ups],
    system_prompt=erp_memo_system_prompt,
    checkpointer=checkpointer,
    store=store,
    context_schema=UserStaticContext
)

## Thread 1：寫入 ERP 工作備忘錄

In [6]:
# inputs for the invocation
inputState = {
    "messages" : [{
                "role": "user",
                "content": (
                    "備忘事項：採購單 B2048 "
                    "等供應商補報價後再送審。"
                )
            }]
}

inputConfig = {
    "configurable": {
        "thread_id": "PO-B2048"
    }
}

userStaticContext = UserStaticContext(user_id="user_123")

In [8]:
response_1 = agent.invoke(
    input=inputState,
    config=inputConfig,
    context=userStaticContext
)

In [9]:
# print the response
from pprint import pp 

pp(response_1)

{'messages': [HumanMessage(content='備忘事項：採購單 B2048 等供應商補報價後再送審。', additional_kwargs={}, response_metadata={}, id='abae2430-85c9-4f10-bf99-1dbc8085c676'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 637, 'prompt_tokens': 363, 'total_tokens': 1000, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 576, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-ECgoegmHxcYCYgqlsp9FYDlSwMBpK', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fff37-0e13-7d50-adef-787ee5d4c7fe-0', tool_calls=[{'name': 'save_erp_follow_up', 'args': {'document_type': 'purchase_order', 'document_id': 'B2048', 'note': '備忘事項：採購單 B2048 等供應商補報價後再送審。'}, 'id': 'c

## Thread 2：讀取 ERP 工作備忘錄

In [10]:
inputState_2 = {
        "messages": [
            {
                "role": "user",
                "content": "我目前有哪些採購事項需要追蹤？"
            }
        ]
    }

inputConfig_2 = {
    "configurable": {
        "thread_id": "follow-up-list"
    }
}

In [11]:
# invoke the agent to get the follow-up list
response_2 = agent.invoke(
    input=inputState_2,
    config=inputConfig_2,
    context=userStaticContext
)

In [14]:
# print the response
pp(response_2)

{'messages': [HumanMessage(content='我目前有哪些採購事項需要追蹤？', additional_kwargs={}, response_metadata={}, id='57a8a8dd-4051-4e8a-b2e4-ea1392ef9639'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 279, 'prompt_tokens': 351, 'total_tokens': 630, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-ECgpRqbpyemQ3cDVhHMoItwN14e2L', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fff37-d380-7e82-9f96-c67dbdc5b9f0-0', tool_calls=[{'name': 'get_erp_follow_ups', 'args': {}, 'id': 'call_gJw2Kpi85JeVelzYod2tFFMt', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 